# Diagnóstico del ciclo externo

**Objetivo:** aislar por qué el ciclo sobre `dataset-new-hez` da malos resultados.

**Hipótesis principal:** el dataset externo calcula $\langle s_z \rangle$ promedio de 5 capas,
mientras que DDPM y Xception se entrenaron sobre la capa central única.
Esto introduce un *domain shift* en las imágenes que puede afectar:
- (A) las métricas de imagen (MSE/SSIM/FFT) → siempre malas si hay domain shift
- (B) las métricas de regresión (R²/MAE de θ) → malas si además los parámetros
  están fuera de distribución o el DDPM genera ruido para esos valores

## Cuatro pruebas
| Prueba | Pregunta | Falla implica |
|--------|----------|---------------|
| P1 · Rangos de parámetros | ¿Están los params externos dentro del rango del scaler? | Los modelos reciben condiciones fuera de distribución |
| P2 · Estadísticas de imagen | ¿Las imágenes externas tienen distinta distribución de píxeles? | Domain shift confirmado (5-capas vs 1-capa) |
| P3 · Xception sobre imágenes ORIGINALES externas (sin DDPM) | ¿El modelo inverso generaliza a imágenes de 5 capas? | La capa de imagen es el cuello de botella |
| P4 · DDPM con parámetros INTERNOS (sanity check) | ¿El ciclo funciona si le damos params del dataset original? | El DDPM/ciclo en sí está roto, no es el domain shift |

## 0 · Setup y descarga

In [ ]:
import subprocess, sys, os

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("kaggle"); pip("scikit-image")

from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as fp:
        fp.write(list(uploaded.values())[0])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

WORKDIR  = "/content"
DATA_DIR = f"{WORKDIR}/kaggle_data"
os.makedirs(DATA_DIR, exist_ok=True)

datasets = [
    "carloscanamejoy/dataset-spines-united-v2",
    "carloscanamejoy/dataset-new-hez",
    "carloscanamejoy/weights-xception-model",
    "carloscanamejoy/weights-models",
]
for ds in datasets:
    name = ds.split("/")[1]
    dest = f"{DATA_DIR}/{name}"
    if not os.path.exists(dest):
        print(f"Descargando {ds} ...")
        subprocess.check_call(["kaggle","datasets","download","-d",ds,"-p",dest,"--unzip"])
    else:
        print(f"  Ya existe: {dest}")

## 1 · Configuración y carga

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# ── Rutas ─────────────────────────────────────────────────────────────────────
DATASET_ORIG_PATH = Path(f"{DATA_DIR}/dataset-spines-united-v2/dataset_unificado_v2.npz")
DATASET_EVAL_PATH = Path(f"{DATA_DIR}/dataset-new-hez/dataset-hez-profdario.npz")
XCEPTION_WEIGHTS  = Path(f"{DATA_DIR}/weights-xception-model/modelo_xception_fulldatabaseV3100.h5")
DDPM_CHECKPOINT   = Path(f"{DATA_DIR}/weights-models/ddpm_spines_final_39crop.pt")

for p in [DATASET_ORIG_PATH, DATASET_EVAL_PATH, XCEPTION_WEIGHTS, DDPM_CHECKPOINT]:
    print(f"[{'OK' if p.exists() else 'FALTA'}]  {p}")

PARAM_NAMES = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]
GRID        = 39
IMG_SIZE    = 40
RD_PIXELS   = 18.3
SEED        = 42

# ── Cargar datasets ───────────────────────────────────────────────────────────
data_orig = np.load(DATASET_ORIG_PATH, mmap_mode="r")
imgs_orig   = data_orig["img"]            # (N_o, 39, 39)  o  (N_o, 39, 39, 1)
params_orig = np.asarray(data_orig["params"])
N_orig = imgs_orig.shape[0]

data_ext  = np.load(DATASET_EVAL_PATH, mmap_mode="r")
imgs_ext    = data_ext["img"]             # (N_e, 39, 39)  o  (N_e, 39, 39, 1)
params_ext  = np.asarray(data_ext["params"])
N_ext = imgs_ext.shape[0]

print(f"\nDataset ORIGINAL : {N_orig:,} muestras  imgs={imgs_orig.shape}")
print(f"Dataset EXTERNO  : {N_ext:,}  muestras  imgs={imgs_ext.shape}")

# ── Scalers (ajustados SOLO sobre train del dataset original) ─────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

all_idx = np.arange(N_orig)
idx_tr_pool, _, _, _ = train_test_split(all_idx, params_orig, test_size=0.15, random_state=SEED)
idx_tr, _,   _, _ = train_test_split(idx_tr_pool, params_orig[idx_tr_pool], test_size=0.1765, random_state=SEED)

scaler_inv  = MinMaxScaler().fit(params_orig[idx_tr])

rng = np.random.RandomState(SEED)
sub = rng.choice(N_orig, size=N_orig, replace=False)
idx_all = np.arange(N_orig)
idx_tr_ddpm, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=SEED)
scaler_ddpm = MinMaxScaler().fit(params_orig[idx_tr_ddpm])

# mn/mx para DDPM (calculado sobre imágenes paddeadas del train)
sample_imgs = np.asarray(imgs_orig[idx_tr_ddpm[:5000]]).astype(np.float32)
if sample_imgs.ndim == 4: sample_imgs = sample_imgs[..., 0]
sample_40 = np.pad(sample_imgs, ((0,0),(0,1),(0,1)), mode='reflect')
mn, mx = float(sample_40.min()), float(sample_40.max())
print(f"\nDDPM image norm: mn={mn:.4f}  mx={mx:.4f}")

---
## P1 · Rangos de parámetros
**Pregunta:** ¿Los parámetros del dataset externo están dentro del rango con que se ajustaron los scalers?

Si algún parámetro del externo está fuera de `[data_min_, data_max_]` del scaler,
el DDPM recibe condiciones normalizadas fuera de `[0,1]` → generación fuera de distribución.

In [ ]:
print("=" * 80)
print("P1 · Rangos de parámetros: original (train) vs externo")
print("=" * 80)
print(f"  {'param':<6}  {'scaler [min, max]':<26}  {'externo [min, max]':<26}  {'out-of-range?':<14}")
print("-" * 80)

oor_params = []
for j, name in enumerate(PARAM_NAMES):
    s_lo  = scaler_ddpm.data_min_[j]
    s_hi  = scaler_ddpm.data_max_[j]
    e_lo  = params_ext[:, j].min()
    e_hi  = params_ext[:, j].max()
    oor_lo = e_lo < s_lo - 1e-9
    oor_hi = e_hi > s_hi + 1e-9
    flag   = ("LOW " if oor_lo else "") + ("HIGH" if oor_hi else "")
    if flag: oor_params.append(name)
    print(f"  {name:<6}  [{s_lo:8.4f}, {s_hi:8.4f}]          [{e_lo:8.4f}, {e_hi:8.4f}]          {flag if flag else 'OK'}")

print()
if oor_params:
    print(f"⚠  Parámetros fuera de rango del scaler: {oor_params}")
    print("   Estos valores se normalizan fuera de [0,1] → el DDPM recibe condiciones")
    print("   fuera de su distribución de entrenamiento.")
else:
    print("✓  Todos los parámetros externos están dentro del rango del scaler.")
    print("   → El problema NO es out-of-range en los parámetros.")

# Fracción de muestras fuera de rango por parámetro
print("\nFracción de muestras del externo con parámetros fuera del rango del scaler:")
for j, name in enumerate(PARAM_NAMES):
    s_lo = scaler_ddpm.data_min_[j]
    s_hi = scaler_ddpm.data_max_[j]
    n_oor = np.sum((params_ext[:, j] < s_lo) | (params_ext[:, j] > s_hi))
    pct = 100 * n_oor / N_ext
    if pct > 0:
        print(f"  {name}: {n_oor}/{N_ext} = {pct:.1f}%")

In [ ]:
# Histogramas superpuestos de cada parámetro
fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for j, (ax, name) in enumerate(zip(axes.ravel(), PARAM_NAMES)):
    bins = np.linspace(
        min(params_orig[:, j].min(), params_ext[:, j].min()),
        max(params_orig[:, j].max(), params_ext[:, j].max()),
        50
    )
    ax.hist(params_orig[:, j], bins=bins, alpha=0.55, color="steelblue",
            density=True, label="Original")
    ax.hist(params_ext[:, j],  bins=bins, alpha=0.55, color="tomato",
            density=True, label="Externo")
    ax.axvline(scaler_ddpm.data_min_[j], color="k", lw=1, ls="--", alpha=0.5)
    ax.axvline(scaler_ddpm.data_max_[j], color="k", lw=1, ls="--", alpha=0.5, label="Scaler range")
    ax.set_title(name); ax.legend(fontsize=7)
fig.suptitle("P1 · Distribución de parámetros: original (azul) vs externo (rojo)\n"
             "Líneas negras = rango del scaler DDPM",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_p1_param_distributions.png")
plt.show()
print("Guardado: diag_p1_param_distributions.png")

---
## P2 · Estadísticas de imagen
**Pregunta:** ¿La distribución de píxeles del dataset externo es diferente a la del dataset original?

**Hipótesis principal:** el externo usa $\langle s_z \rangle$ promedio de 5 capas → imágenes
más suaves, menor contraste, distribución de píxeles diferente.
Esto hace que la comparación DDPM-gen vs imagen-original sea injusta a nivel de píxel.

In [ ]:
# Muestra aleatoria de imágenes de ambos datasets
rng2 = np.random.RandomState(0)
N_SAMPLE = min(5000, N_orig, N_ext)

idx_o = rng2.choice(N_orig, N_SAMPLE, replace=False)
idx_e = rng2.choice(N_ext,  N_SAMPLE, replace=False)

imgs_o = np.asarray(imgs_orig[idx_o]).astype(np.float32)
imgs_e = np.asarray(imgs_ext [idx_e]).astype(np.float32)

if imgs_o.ndim == 4: imgs_o = imgs_o[..., 0]
if imgs_e.ndim == 4: imgs_e = imgs_e[..., 0]

# Estadísticas globales
print("=" * 60)
print("P2 · Estadísticas de imagen (píxeles en [0,1])")
print("=" * 60)
stats = {}
for label, imgs in [("Original", imgs_o), ("Externo", imgs_e)]:
    flat = imgs.ravel()
    stats[label] = {
        "mean": flat.mean(), "std": flat.std(),
        "min":  flat.min(),  "max": flat.max(),
        "p5":   np.percentile(flat, 5), "p95": np.percentile(flat, 95),
    }
    print(f"  {label:<10}  mean={stats[label]['mean']:.4f}  std={stats[label]['std']:.4f}  "
          f"min={stats[label]['min']:.4f}  max={stats[label]['max']:.4f}  "
          f"p5={stats[label]['p5']:.4f}  p95={stats[label]['p95']:.4f}")

diff_mean = abs(stats["Original"]["mean"] - stats["Externo"]["mean"])
diff_std  = abs(stats["Original"]["std"]  - stats["Externo"]["std"])
print(f"\n  Δmean = {diff_mean:.4f}  {'⚠ significativo' if diff_mean > 0.05 else '✓ pequeño'}")
print(f"  Δstd  = {diff_std:.4f}  {'⚠ significativo (distinta variabilidad)' if diff_std > 0.05 else '✓ pequeño'}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histograma de píxeles
ax = axes[0]
ax.hist(imgs_o.ravel(), bins=100, alpha=0.55, color="steelblue",
        density=True, label="Original (1 capa)")
ax.hist(imgs_e.ravel(), bins=100, alpha=0.55, color="tomato",
        density=True, label="Externo (5 capas?)")
ax.set_xlabel("Valor de píxel [0,1]")
ax.set_ylabel("Densidad")
ax.set_title("Histograma de píxeles")
ax.legend()

# Media de imagen por muestra
ax = axes[1]
ax.hist(imgs_o.mean(axis=(1,2)), bins=60, alpha=0.55, color="steelblue", density=True, label="Original")
ax.hist(imgs_e.mean(axis=(1,2)), bins=60, alpha=0.55, color="tomato", density=True, label="Externo")
ax.set_xlabel("Media de imagen")
ax.set_title("Distribución de la media por imagen")
ax.legend()

# Std de imagen por muestra
ax = axes[2]
ax.hist(imgs_o.std(axis=(1,2)), bins=60, alpha=0.55, color="steelblue", density=True, label="Original")
ax.hist(imgs_e.std(axis=(1,2)), bins=60, alpha=0.55, color="tomato", density=True, label="Externo")
ax.set_xlabel("Std de imagen")
ax.set_title("Distribución del contraste (std) por imagen")
ax.legend()

fig.suptitle("P2 · Distribución de píxeles: original vs externo", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_p2_pixel_distributions.png")
plt.show()
print("Guardado: diag_p2_pixel_distributions.png")

In [ ]:
# Galería visual: 8 imágenes de cada dataset
N_SHOW = 8
fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 2.2, 5))

for col in range(N_SHOW):
    axes[0, col].imshow(imgs_o[col], cmap="jet", vmin=0, vmax=1, origin="lower")
    axes[0, col].axis("off")
    axes[1, col].imshow(imgs_e[col], cmap="jet", vmin=0, vmax=1, origin="lower")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Original\n(capa central)", fontsize=9)
axes[1, 0].set_ylabel("Externo\n(¿5 capas?)",     fontsize=9)
fig.suptitle("P2 · Inspección visual: ¿las texturas se ven diferentes?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_p2_visual_comparison.png")
plt.show()
print("Guardado: diag_p2_visual_comparison.png")
print()
print("INTERPRETACIÓN:")
print("  Si las imágenes externas se ven más suaves/borrosas → domain shift confirmado.")
print("  Si se ven similares → el problema es otro (parámetros o el DDPM en sí).")

---
## P3 · Xception sobre imágenes ORIGINALES del externo (sin DDPM)

**Pregunta:** Si le damos a Xception las imágenes del dataset externo directamente,
¿recupera bien los parámetros?

Esto prueba si el modelo inverso generaliza a imágenes de 5 capas.

- Si **sí** recupera bien → Xception generaliza, el problema está en que el DDPM genera
  imágenes de 1 capa y Xception las interpreta bien, pero la comparación pixel-a-pixel
  con las originales de 5 capas es la causa del mal SSIM/MSE.
  
- Si **no** recupera bien → las imágenes de 5 capas tienen texturas lo suficientemente
  distintas como para confundir a Xception también.

In [ ]:
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass

from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dense,
                                     BatchNormalization, Dropout)
from tensorflow.keras.models import Model

def build_xception(n_outputs=8):
    inp  = Input(shape=(224, 224, 3))
    base = Xception(weights=None, include_top=False, input_tensor=inp)
    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization(name="batch_normalization_4")(x)
    x = Dropout(0.4, name="dropout")(x)
    x = Dense(256, activation="relu", name="dense")(x)
    x = BatchNormalization(name="batch_normalization_5")(x)
    x = Dropout(0.3, name="dropout_1")(x)
    out = Dense(n_outputs, activation="linear", name="dense_1")(x)
    return Model(inp, out)

with tf.device("/cpu:0"):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_WEIGHTS)
print(f"Xception cargado. Parámetros: {xception_model.count_params():,}")

def xception_predict_batch(imgs_39, batch_size=64):
    """(N,39,39) o (N,39,39,1) → parámetros físicos (N,8)."""
    imgs = np.asarray(imgs_39, dtype=np.float32)
    if imgs.ndim == 4: imgs = imgs[..., 0]
    results = []
    with tf.device("/cpu:0"):
        for start in range(0, len(imgs), batch_size):
            chunk = imgs[start:start+batch_size][..., None]  # (B,39,39,1)
            chunk = tf.image.resize(chunk, (224, 224))        # (B,224,224,1)
            chunk = tf.image.grayscale_to_rgb(chunk)          # (B,224,224,3)
            y_norm = xception_model.predict(chunk, verbose=0)
            results.append(scaler_inv.inverse_transform(y_norm))
    return np.concatenate(results, axis=0)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

N_P3 = min(1000, N_ext)
idx_p3 = np.random.RandomState(1).choice(N_ext, N_P3, replace=False)

print(f"P3 · Xception sobre imágenes ORIGINALES del externo (n={N_P3})...")
imgs_p3   = np.asarray(imgs_ext[idx_p3]).astype(np.float32)
params_p3 = params_ext[idx_p3]

y_pred_p3 = xception_predict_batch(imgs_p3)

print("\n  Resultados (imagen original externo → Xception → θ̂):")
print(f"  {'param':<6}  {'R²':>8}  {'MAE':>10}")
r2_list, mae_list = [], []
for j, name in enumerate(PARAM_NAMES):
    yt = params_p3[:, j]; yp = y_pred_p3[:, j]
    r2  = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)
    r2_list.append(r2); mae_list.append(mae)
    flag = "  ✓" if r2 > 0.7 else ("  ~" if r2 > 0.3 else "  ✗")
    print(f"  {name:<6}  {r2:8.4f}  {mae:10.4f}{flag}")

print(f"\n  R² promedio:  {np.mean(r2_list):.4f}")
print(f"  MAE promedio: {np.mean(mae_list):.4f}")
print()
print("INTERPRETACIÓN:")
print("  R² > 0.7 para los 3 parámetros identificables (T, Jex2, KDM)")
print("  → Xception SÍ generaliza a imágenes externas.")
print("  → El ciclo falla porque el DDPM genera imágenes de 1 capa,")
print("    que lucen DIFERENTES a las originales de 5 capas en métricas pixel.")
print()
print("  R² bajo incluso para T, Jex2, KDM")
print("  → Las imágenes de 5 capas son suficientemente distintas")
print("    para confundir a Xception también.")

In [ ]:
# Scatter true vs pred para los 3 parámetros más identificables
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["T", "Jex2", "KDM"]):
    j  = PARAM_NAMES.index(name)
    yt = params_p3[:, j]; yp = y_pred_p3[:, j]
    ax.scatter(yt, yp, s=8, alpha=0.4, color="tomato")
    lo = min(yt.min(), yp.min()); hi = max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"{name}  R²={r2_score(yt,yp):.3f}  MAE={mean_absolute_error(yt,yp):.4f}")
    ax.set_xlabel("θ real"); ax.set_ylabel("θ̂ (Xception directo)")
fig.suptitle("P3 · Xception sobre imágenes ORIGINALES del externo", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_p3_xception_on_external.png")
plt.show()
print("Guardado: diag_p3_xception_on_external.png")

---
## P4 · Sanity check del DDPM con parámetros INTERNOS

**Pregunta:** ¿El ciclo funciona correctamente cuando le damos parámetros
del dataset ORIGINAL (en los que se entrenó)?

Si el ciclo funciona aquí pero no con el externo → el problema es el domain shift
de parámetros o imágenes externas, no un fallo del DDPM en sí.

Si el ciclo también falla aquí → hay un bug en el DDPM o en el pipeline del ciclo.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Reconstruir DDPM (misma arquitectura que ciclo_external_dataset.ipynb) ────

def sinusoidal_embedding(t, dim):
    half  = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half-1))
    args  = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
    def forward(self, t, cond):
        return self.t_mlp(sinusoidal_embedding(t, self.t_mlp[0].in_features)) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch);  self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch); self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip     = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, emb):
        h = F.silu(self.norm1(x)); h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h)); h = self.dropout(h); h = self.conv2(h)
        return h + self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv  = nn.Conv2d(ch, ch*3, 1); self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        B, C, H, W = x.shape; h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B, C, -1); k = k.reshape(B, C, -1); v = v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1,2), k) / math.sqrt(C), dim=-1)
        return x + self.proj(torch.bmm(v, attn.transpose(1,2)).reshape(B, C, H, W))

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1,2,4),
                 cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch*m for m in ch_mults]
        self.emb    = TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)
        self.down_blocks  = nn.ModuleList(); self.down_samples = nn.ModuleList(); self.skip_channels = []
        in_ch = chs[0]
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn   = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.up_blocks  = nn.ModuleList(); self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i+1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch+skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)
    def forward(self, x, t, cond):
        emb = self.emb(t, cond); h = self.conv_in(x); skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb); skips.append(h); h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, sk in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, sk], dim=1); h = rb1(h, emb); h = rb2(h, emb); h = us(h)
        return self.conv_out(F.silu(self.norm_out(h)))

class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="linear", device="cpu"):
        self.T = T
        if schedule == "cosine":
            steps = T + 1; s = 0.008
            x  = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x/T)+s)/(1+s)*math.pi*0.5)**2
            ac = ac / ac[0]
            betas = (1 - ac[1:]/ac[:-1]).clamp(max=0.999)
        else:
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        ac     = torch.cumprod(alphas, dim=0)
        self.betas = betas; self.alphas = alphas; self.alphas_cumprod = ac
        self.sqrt_alphas_cumprod           = ac.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - ac).sqrt()

@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    model.eval()
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_t  = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps  = model(x, t_t, cond)
        sa   = scheduler.sqrt_alphas_cumprod[t_val]
        s1a  = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0   = ((x - s1a*eps) / sa).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            x = scheduler.sqrt_alphas_cumprod[t_prev]*x0 + scheduler.sqrt_one_minus_alphas_cumprod[t_prev]*eps
        else:
            x = x0
    return x

# Cargar checkpoint
ckpt = torch.load(DDPM_CHECKPOINT, map_location=DEVICE, weights_only=False)
hp   = ckpt["hyperparams"]
print(f"Hiperparámetros DDPM: {hp}")

ddpm_model = ConditionalUNet(
    img_channels=1, base_ch=hp["base_ch"], ch_mults=(1,2,4),
    cond_dim=8, emb_dim=hp["cond_emb_dim"], dropout=0.0,
).to(DEVICE)

if "ema" in ckpt and ckpt["ema"] is not None:
    ddpm_model.load_state_dict(ckpt["ema"]); print("EMA cargado.")
else:
    ddpm_model.load_state_dict(ckpt["model"]); print("Pesos raw cargados.")
ddpm_model.eval()

scheduler = DDPMScheduler(T=1000, beta_start=1e-4, beta_end=0.02,
                           schedule=hp["beta_schedule"], device=DEVICE)
print("DDPM listo.")

In [ ]:
# Tomar parámetros del TEST SET del dataset ORIGINAL
all_idx_o = np.arange(N_orig)
_, idx_te_o, _, _ = train_test_split(all_idx_o, params_orig, test_size=0.15, random_state=SEED)

N_P4 = min(500, len(idx_te_o))
idx_p4 = np.random.RandomState(2).choice(idx_te_o, N_P4, replace=False)
params_p4 = params_orig[idx_p4].astype(np.float32)

print(f"P4 · Sanity check con {N_P4} parámetros del test set ORIGINAL...")

# Generar imágenes con DDPM
BATCH = 64
imgs_gen_p4 = []
for start in range(0, N_P4, BATCH):
    end   = min(start + BATCH, N_P4)
    cond  = scaler_ddpm.transform(params_p4[start:end])
    cond_t = torch.tensor(cond, dtype=torch.float32, device=DEVICE)
    x = fast_sample(ddpm_model, cond_t, scheduler, n_steps=100, img_size=40)
    imgs_gen_p4.append(x.cpu().numpy()[:, 0])
imgs_gen_p4 = np.concatenate(imgs_gen_p4, axis=0)  # (N_P4, 40, 40)

# Crop y pasar por Xception
imgs_for_xcep = imgs_gen_p4[:, :39, :39]                        # (N_P4, 39, 39)
imgs_for_xcep = (imgs_for_xcep + 1.0) / 2.0 * (mx - mn) + mn   # físico
y_pred_p4     = xception_predict_batch(imgs_for_xcep)

print("\n  Resultados (parámetros internos → DDPM → imagen → Xception → θ̂):")
print(f"  {'param':<6}  {'R²':>8}  {'MAE':>10}")
for j, name in enumerate(PARAM_NAMES):
    yt  = params_p4[:, j]; yp = y_pred_p4[:, j]
    r2  = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)
    flag = "  ✓" if r2 > 0.7 else ("  ~" if r2 > 0.3 else "  ✗")
    print(f"  {name:<6}  {r2:8.4f}  {mae:10.4f}{flag}")

print()
print("INTERPRETACIÓN:")
print("  R² alto para T, Jex2, KDM → el ciclo funciona para parámetros internos.")
print("  Si P4 funciona pero P3 no → el problema es el domain shift de las imágenes externas.")
print("  Si P4 también falla → hay un bug en el pipeline de ciclo.")

In [ ]:
# Scatter true vs pred para los parámetros identificables (sanity check)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["T", "Jex2", "KDM"]):
    j  = PARAM_NAMES.index(name)
    yt = params_p4[:, j]; yp = y_pred_p4[:, j]
    ax.scatter(yt, yp, s=8, alpha=0.4, color="steelblue")
    lo = min(yt.min(), yp.min()); hi = max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"{name}  R²={r2_score(yt,yp):.3f}  MAE={mean_absolute_error(yt,yp):.4f}")
    ax.set_xlabel("θ real (interno)"); ax.set_ylabel("θ̂ (DDPM → Xception)")
fig.suptitle("P4 · Sanity check: ciclo con parámetros INTERNOS (¿funciona el ciclo?)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("diag_p4_sanity_internal.png")
plt.show()
print("Guardado: diag_p4_sanity_internal.png")

---
## Resumen del diagnóstico

In [ ]:
print("=" * 70)
print("RESUMEN DEL DIAGNÓSTICO")
print("=" * 70)

print("""
P1 · ¿Parámetros fuera de rango del scaler?
   Ver tabla arriba. Si hay parámetros con flag ≠ OK, el DDPM recibe
   condiciones fuera de su distribución de entrenamiento.

P2 · ¿Las imágenes externas tienen distinta distribución de píxeles?
   Compara los histogramas y las galerías visuales.
   Si se ven más suaves/borrosas → domain shift confirmado (5-capas vs 1-capa).
   Esto explica por sí solo los malos MSE/SSIM incluso con un DDPM perfecto.

P3 · ¿Xception generaliza a imágenes externas directas?
   Si R²(T) y R²(KDM) > 0.7 en P3:
     → El modelo inverso SÍ generaliza.
     → Los malos R² del ciclo se deben a que el DDPM genera imágenes de
       1 capa que Xception interpreta bien, pero la COMPARACIÓN con las
       originales de 5 capas (image metrics) es injusta.
     → CONCLUSIÓN: el ciclo θ→DDPM→Xception→θ̂ FUNCIONA en el externo,
       pero no tiene sentido comparar pixel-a-pixel DDPM(1 capa) vs original(5 capas).

   Si R²(T) y R²(KDM) < 0.5 en P3:
     → Las imágenes de 5 capas tienen texturas lo suficientemente distintas.
     → Xception no puede recuperar parámetros de esas imágenes.
     → Para que el ciclo funcione en el externo, se necesita reentrenar
       con imágenes de 5 capas, o hacer data augmentation.

P4 · ¿El ciclo funciona con parámetros internos?
   Si R²(T,Jex2,KDM) > 0.7 en P4 pero no en el ciclo externo:
     → El DDPM/ciclo en sí está correcto.
     → El problema es específico del dataset externo (P1 o P2).

   Si P4 también falla:
     → Hay un bug en el pipeline (scaler, padding, crop).
     → Revisar que los pesos EMA se carguen correctamente.
""")